# Lab 08a — Food Analyzer and Recipe Builder
## Revised Anaconda notebook: repair the setup before launching the app

Your uploaded outputs show **Python 3.14.6**, a failed **NumPy source build**, and a
failed **tokenizers 0.20.3 build**. The tokenizers error explicitly says Python 3.14
exceeds that build's supported version. Consequently, `transformers` never installs.
The old Cell 1 still prints “Dependencies installed” after those failures, so later
cells send you back to an installation that did not finish.

**The solution here is a separate Python 3.11 environment and a compatible pinned
package set.** Cell 1 checks actual installation results and imports CLIP immediately.
Cell 4 tests image preprocessing and model inference before Cell 6 starts Gradio.

### Do this once in macOS Terminal

Run these two commands in Terminal, one at a time. Wait for each to finish:

```bash
conda create --name nutritionist311 python=3.11 pip ipykernel -y
conda run -n nutritionist311 python -m ipykernel install --user --name nutritionist311 --display-name "Python (Nutritionist 3.11)"
```

If `conda` is not found in Terminal, use `/opt/anaconda3/bin/conda` in place of
`conda` in both commands; that is the Anaconda installation shown in your outputs.
If you already created `nutritionist311` with Python 3.11, skip the first command.

### Open and run this notebook

1. Open this file in Jupyter through Anaconda.
2. Choose **Kernel → Change Kernel → Python (Nutritionist 3.11)**. If it is missing,
   refresh or reopen Jupyter after running the registration command above.
3. Run **Cell 1**. It must print Python **3.11.x** and **SETUP PASSED**.
   If it explicitly requests a restart, choose **Kernel → Restart Kernel**, then
   **Run All**. Matching versions will not be installed again.
4. In LM Studio, load a **chat/instruct** model and start its local server on
   port **1234**. Run **Cells 2–6 in order**. Cell 2 uses the model ID reported by
   the server. If several models are listed, paste the desired ID into `MODEL_ID`.
5. Wait for **CLIP PASSED** in Cell 4; the first run needs internet access and
   downloads roughly **600 MB** of model weights. Cell 6 opens the local app.
   Upload a photo, then click **Analyze & Create Recipe**.

All Python package installs target the selected notebook kernel. This lab does not
need `torchvision`, `torchaudio`, or `timm`. PyTorch remains necessary for CLIP.
The first run may take several minutes. Run the notebook locally on the same
computer as LM Studio so `localhost:1234` refers to the correct machine.

**Reading the results:** CLIP ranks candidate food labels; it does not independently
detect and weigh every ingredient. The top three are suggestions to check visually.
Nutrition uses the original small teaching lookup and assumed portions; totals are
for the whole dish. Foods missing from that lookup are reported as excluded.
A generated recipe may change ingredients or quantities, so its nutrition is not
automatically represented by the photo's nutrition table.


In [ ]:
# === Cell 1: Reliable setup for this notebook's Python 3.11 kernel ===
# This cell installs into THIS kernel, checks pip's exit status, and then
# imports CLIP immediately. It does not assume that printing means success.
import os
import sys
import subprocess
import importlib
import importlib.metadata as metadata
from packaging.requirements import Requirement

SETUP_READY = False
print('Notebook Python:', sys.version.split()[0])
print('Executable:', sys.executable)

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        'Select the kernel "Python (Nutritionist 3.11)".\n'
        'Your current kernel is not Python 3.11. Follow the two Terminal '
        'commands in the instructions above, then change the notebook kernel.\n'
        'Installing again into the Python 3.14 base environment will not '
        'repair the original dependency build failures.'
    )

# A rerun in the same process must not bypass a requested restart.
if globals().get('_SETUP_RESTART_REQUIRED', False):
    raise RuntimeError(
        'Packages changed while this kernel had libraries loaded. '
        'Choose Kernel > Restart Kernel, then Run All.'
    )

REQUIREMENTS = ['torch==2.8.0', 'transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface-hub==0.36.0', 'safetensors==0.6.2', 'numpy==1.26.4', 'pandas==2.2.3', 'pillow==11.3.0', 'gradio==5.49.1', 'gradio-client==1.13.3', 'pydantic==2.11.10', 'requests==2.32.5']

def requirements_to_install():
    missing = []
    for text in REQUIREMENTS:
        requirement = Requirement(text)
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            missing.append(text)
            continue
        if not requirement.specifier.contains(installed):
            missing.append(text)
    return missing

needed = requirements_to_install()
if needed:
    print('\nInstalling the compatible lab stack. First run may take several minutes.')
    loaded_before_install = any(
        name in sys.modules
        for name in ('torch', 'transformers', 'tokenizers', 'numpy', 'pandas',
                     'PIL', 'gradio', 'pydantic', 'requests', 'huggingface_hub')
    )
    try:
        # One resolver transaction prevents unrelated, incompatible upgrades.
        # Wheels only: fail clearly instead of trying a C++ or Rust build.
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--upgrade',
             '--only-binary=:all:', '--disable-pip-version-check', *REQUIREMENTS],
            check=True,
        )
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            'Installation failed; setup has stopped. Read the pip error above. '
            'Confirm that the selected kernel is Python (Nutritionist 3.11) '
            'and that internet access is available. Do not continue to the app.'
        ) from error

    importlib.invalidate_caches()
    if loaded_before_install:
        _SETUP_RESTART_REQUIRED = True
        raise RuntimeError(
            'Installation completed, but this process already had packages loaded. '
            'This is the one-time restart step: choose Kernel > Restart Kernel, '
            'then Run All. Cell 1 will skip installation when the versions match.'
        )
else:
    print('\nCompatible package versions already installed; skipping installation.')

# Avoid pulling unrelated TensorFlow/Flax backends into a PyTorch-only lab.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
try:
    import torch
    import numpy as np
    import pandas as pd
    import gradio as gr
    import requests
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor
    assert torch.tensor([1.0, 2.0]).numpy().tolist() == [1.0, 2.0]
except Exception as error:
    raise RuntimeError(
        'The package import check failed BEFORE the app was started. '
        'If packages were just changed, restart the kernel and Run All. '
        f'Active Python: {sys.executable}. Original error: {error}'
    ) from error

remaining = requirements_to_install()
if remaining:
    raise RuntimeError(f'These package versions still do not match: {remaining}')

for package in ('torch', 'transformers', 'tokenizers', 'numpy', 'pandas', 'pillow', 'gradio'):
    print(f'  {package}: {metadata.version(package)}')
SETUP_READY = True
print('\nSETUP PASSED: CLIPModel and CLIPProcessor imported successfully.')
print('Continue to Cell 2. The CLIP weights are downloaded and tested in Cell 4.')


## Cell 2 — Connect to your local LM Studio server
Rerun this cell if you change the server or chat model.


In [ ]:
# === Cell 2: Connect to LM Studio and select an actual server model ID ===
if not globals().get('SETUP_READY', False):
    raise RuntimeError('Complete Cell 1 successfully before running Cell 2.')

import json
import requests

# Start LM Studio on THIS computer, load a chat/instruct model, and enable
# its local server. The notebook runs in local Anaconda, not Google Colab.
LMSTUDIO_BASE_URL = 'http://localhost:1234/v1'
LMSTUDIO_API_KEY = os.environ.get('LMSTUDIO_API_KEY', 'lmstudio-key')
MODEL_ID = ''  # Leave blank for one chat model; otherwise paste a printed ID.
LMSTUDIO_READY = False

headers = {'Authorization': f'Bearer {LMSTUDIO_API_KEY}'}
try:
    response = requests.get(
        f'{LMSTUDIO_BASE_URL.rstrip("/")}/models', headers=headers, timeout=10,
    )
    response.raise_for_status()
    available_models = response.json().get('data', [])
except (requests.RequestException, ValueError) as error:
    raise RuntimeError(
        f'Cannot read models from {LMSTUDIO_BASE_URL}/models. '
        'Open LM Studio, load a chat model, enable its local server on port 1234, '
        'then rerun THIS cell. If authentication is enabled, set LMSTUDIO_API_KEY '
        f'to your LM Studio token. Details: {error}'
    ) from error

model_ids = [item['id'] for item in available_models if item.get('id')]
print('Model IDs reported by LM Studio:')
for model_id in model_ids:
    print(' -', model_id)

if not MODEL_ID:
    # /models can include downloaded models, including embedding models.
    chat_candidates = [model_id for model_id in model_ids if 'embed' not in model_id.lower()]
    if len(chat_candidates) != 1:
        raise RuntimeError(
            'Set MODEL_ID above to the exact ID of the chat/instruct model you '
            'loaded in LM Studio, then rerun Cell 2. '
            f'Found {len(chat_candidates)} possible chat models.'
        )
    MODEL_ID = chat_candidates[0]
if MODEL_ID not in model_ids:
    raise RuntimeError(f'MODEL_ID {MODEL_ID!r} is not one of the IDs printed above.')

# Keep the existing pipeline's environment-variable interface.
os.environ['OPENAI_BASE_URL'] = LMSTUDIO_BASE_URL.rstrip('/')
os.environ['OPENAI_API_KEY'] = LMSTUDIO_API_KEY
os.environ['MODEL'] = MODEL_ID
LMSTUDIO_READY = True
print('\nLM Studio server is reachable. Selected model:', MODEL_ID)


## Cell 3 — Nutrition lookup and shared helpers
The original teaching lookup is retained.


In [ ]:
# === Cell 3: Imports, nutrition database, and helper functions ===

from typing import List, Dict, Any
from PIL import Image
import numpy as np
import pandas as pd

NUTRITION_DB = {
    "salmon": {"calories": 208, "protein_g": 20, "fat_g": 13, "carbs_g": 0, "fiber_g": 0},
    "asparagus": {"calories": 20, "protein_g": 2.2, "fat_g": 0.1, "carbs_g": 3.9, "fiber_g": 2.1},
    "tomato": {"calories": 18, "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 3.9, "fiber_g": 1.2},
    "rice": {"calories": 130, "protein_g": 2.7, "fat_g": 0.3, "carbs_g": 28, "fiber_g": 0.4},
    "pasta": {"calories": 131, "protein_g": 5, "fat_g": 1.1, "carbs_g": 25, "fiber_g": 1.3},
    "avocado": {"calories": 160, "protein_g": 2, "fat_g": 15, "carbs_g": 9, "fiber_g": 7},
    "egg": {"calories": 155, "protein_g": 13, "fat_g": 11, "carbs_g": 1.1, "fiber_g": 0},
    "chicken_breast": {"calories": 165, "protein_g": 31, "fat_g": 3.6, "carbs_g": 0, "fiber_g": 0},
    "beef_steak": {"calories": 271, "protein_g": 25, "fat_g": 19, "carbs_g": 0, "fiber_g": 0},
    "broccoli": {"calories": 34, "protein_g": 2.8, "fat_g": 0.4, "carbs_g": 7, "fiber_g": 2.6},
    "potato": {"calories": 77, "protein_g": 2, "fat_g": 0.1, "carbs_g": 17, "fiber_g": 2.2},
    "spinach": {"calories": 23, "protein_g": 2.9, "fat_g": 0.4, "carbs_g": 3.6, "fiber_g": 2.2},
    "lemon": {"calories": 29, "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 9.3, "fiber_g": 2.8},
    "olive_oil": {"calories": 884, "protein_g": 0, "fat_g": 100, "carbs_g": 0, "fiber_g": 0},
}

CANDIDATE_LABELS = [
    "salmon", "asparagus", "tomato", "rice", "pasta", "avocado", "egg",
    "chicken breast", "beef steak", "broccoli", "potato", "spinach", "lemon",
    "olive oil", "cheeseburger", "pizza", "pancakes", "sushi", "tofu"
]

def to_safe_key(label: str) -> str:
    label = label.lower().strip().replace(" ", "_")
    mapping = {
        "chicken_breast": "chicken_breast",
        "chicken": "chicken_breast",
        "beef_steak": "beef_steak",
        "steak": "beef_steak",
    }
    return mapping.get(label, label)

def scale_nutrition(per100g: Dict[str, float], grams: float) -> Dict[str, float]:
    factor = grams / 100.0
    return {k: round(v * factor, 2) for k, v in per100g.items()}

print("Nutrition lookup loaded. Values are approximate, per 100 grams.")


## Cell 4 — Load CLIP and test it now
Wait for **CLIP PASSED** before continuing. A download error here concerns the model files; it does not mean that `transformers` is missing.


In [ ]:
# === Cell 4: Load and test CLIP BEFORE opening Gradio ===
# First run downloads the official CLIP weights (roughly 600 MB).
# Later runs normally reuse Hugging Face's local cache.
CLIP_READY = False
if not globals().get('SETUP_READY', False):
    raise RuntimeError('Complete Cell 1 successfully before loading CLIP.')

import torch
from transformers import CLIPModel, CLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
print('Loading:', CLIP_MODEL_NAME, '| device:', DEVICE)
try:
    # The official checkpoint provides pytorch_model.bin. Torch 2.8 supports
    # Transformers' guarded loading of these weights.
    clip_model = CLIPModel.from_pretrained(
        CLIP_MODEL_NAME, use_safetensors=False,
    ).to(DEVICE).eval()
    # Slow here means the PIL image processor; torchvision is unnecessary.
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=False)
except Exception as error:
    raise RuntimeError(
        'CLIP imports passed, but loading its weights/processor failed. '
        'Check internet access to https://huggingface.co/openai/clip-vit-base-patch32 '
        'and available disk space, then rerun Cell 4. '
        f'Original error: {type(error).__name__}: {error}'
    ) from error

def get_clip_components():
    return clip_model, clip_processor

def clip_rank_labels(pil_image: Image.Image, candidate_texts: List[str], top_k: int = 3):
    model, processor = get_clip_components()
    inputs = processor(
        text=[f'a photo of {label}' for label in candidate_texts],
        images=pil_image.convert('RGB'),
        return_tensors='pt', padding=True,
    ).to(DEVICE)
    with torch.inference_mode():
        outputs = model(**inputs)
        scores = outputs.logits_per_image.softmax(dim=1).cpu().numpy().flatten()
    ranked = sorted(zip(candidate_texts, scores), key=lambda pair: pair[1], reverse=True)[:top_k]
    return [{'label': label, 'prob': float(score)} for label, score in ranked]

# A synthetic image tests processor -> model -> NumPy without needing a file.
# Its scores are deliberately not interpreted as meaningful food recognition.
_clip_test = clip_rank_labels(
    Image.new('RGB', (224, 224), color='gray'), ['salmon', 'broccoli'], top_k=2,
)
if len(_clip_test) != 2 or not all(np.isfinite(item['prob']) for item in _clip_test):
    raise RuntimeError('CLIP produced an invalid test result; do not launch the app yet.')
CLIP_READY = True
print('CLIP PASSED: preprocessing and a model forward pass completed.')


## Cell 5 — Analysis and recipe functions
Portion overrides accept label names such as `"chicken breast"` or `"chicken_breast"`.


In [ ]:
# === Cell 5: Plain-Python pipeline functions ===

def lmstudio_chat(system_prompt: str, user_prompt: str) -> str:
    """
    Direct OpenAI-compatible HTTP call to LM Studio.
    We fold the system guidance into one user message for broad local-model compatibility.
    """
    url = f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": os.environ["MODEL"],
        "temperature": 0.3,
        "max_tokens": 700,
        "messages": [
            {
                "role": "user",
                "content": (
                    "### System instruction (follow strictly):\n"
                    f"{system_prompt}\n\n"
                    "### User request:\n"
                    f"{user_prompt}"
                ),
            }
        ],
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=(10, 180))
        resp.raise_for_status()
        data = resp.json()
        content = data['choices'][0]['message']['content']
        if not isinstance(content, str) or not content.strip():
            raise ValueError('The chat model returned an empty text response.')
        return content
    except (requests.RequestException, ValueError, KeyError, IndexError, TypeError) as error:
        raise RuntimeError(
            'LM Studio recipe generation failed. Check that its chat model '
            'is loaded, then rerun Cell 2 if the server/model changed. '
            f'Details: {error}'
        ) from error

def detect_foods(image: Image.Image, top_k: int = 3) -> List[Dict[str, Any]]:
    return clip_rank_labels(image, CANDIDATE_LABELS, top_k=top_k)

def estimate_nutrition(detections: List[Dict[str, Any]], grams_map: Dict[str, float]) -> Dict[str, Any]:
    items = []
    skipped = []
    totals = {"calories": 0, "protein_g": 0, "fat_g": 0, "carbs_g": 0, "fiber_g": 0}

    for d in detections:
        raw_label = d["label"]
        key = to_safe_key(raw_label)
        grams = float(grams_map.get(raw_label, grams_map.get(key, 0)))
        if key not in NUTRITION_DB:
            skipped.append(raw_label)
            continue
        if grams <= 0:
            continue

        scaled = scale_nutrition(NUTRITION_DB[key], grams)
        items.append({"ingredient": raw_label, "grams": grams, **scaled})
        for metric in totals:
            totals[metric] += scaled[metric]

    totals = {k: round(v, 2) for k, v in totals.items()}
    return {"items": items, "totals": totals, "skipped": skipped}

def build_recipe(labels: List[str], nutrition: Dict[str, Any], user_prefs: str, servings: int, calories_goal: int) -> str:
    system = "You are a professional chef and nutrition coach. Be concise, friendly, and practical."
    user = f"""
Possible ingredients suggested by CLIP (confirm visually): {labels}
Estimated totals for listed portions (whole dish): {nutrition['totals']}
Foods with no nutrition lookup, excluded from totals: {nutrition.get('skipped', [])}
Portions are assumptions, not measurements from the photo.
User preferences/restrictions: {user_prefs}
Target calories per serving: ~{calories_goal}
Servings: {servings}

Write:
1) A brief 2-sentence assessment of the plate and nutrition.
2) A personalized recipe (ingredients list in grams + step-by-step instructions).
3) One tip to adjust calories up or down while keeping protein reasonable.
"""
    return lmstudio_chat(system, user)

def analyze_food_image(
    image: Image.Image,
    user_prefs: str,
    servings: int = 2,
    calories_goal: int = 600,
    grams_overrides: Dict[str, float] | None = None,
) -> Dict[str, Any]:
    detections = detect_foods(image, top_k=3)
    labels = [d["label"] for d in detections]

    default_grams = {
        label: 150.0 if any(word in label for word in ["salmon", "steak", "chicken"]) else 80.0
        for label in labels
    }
    normalized_overrides = {}
    if grams_overrides is not None:
        if not isinstance(grams_overrides, dict):
            raise ValueError('Portion overrides must be a JSON object.')
        valid_keys = {to_safe_key(label) for label in CANDIDATE_LABELS}
        for label, grams in grams_overrides.items():
            if not isinstance(label, str) or to_safe_key(label) not in valid_keys:
                raise ValueError(f'Unknown food label in portion overrides: {label!r}')
            if isinstance(grams, bool) or not isinstance(grams, (int, float)):
                raise ValueError(f'Portion for {label} must be a number of grams.')
            if not np.isfinite(grams) or grams < 0:
                raise ValueError(f'Portion for {label} must be finite and nonnegative.')
            normalized_overrides[to_safe_key(label)] = float(grams)
    grams_map = {
        label: normalized_overrides.get(to_safe_key(label), grams)
        for label, grams in default_grams.items()
    }
    nutrition = estimate_nutrition(detections, grams_map)
    recipe_text = build_recipe(labels, nutrition, user_prefs, servings, calories_goal)

    return {"detections": detections, "nutrition": nutrition, "recipe_text": recipe_text}
# Table rendering also works in the optional test without launching Gradio.
def nutrition_df(nutrition: Dict[str, Any]) -> pd.DataFrame:
    items = nutrition.get('items', [])
    total_label = 'PARTIAL TOTAL' if nutrition.get('skipped') else 'TOTAL'
    total_row = {
        'ingredient': total_label,
        'grams': round(sum(item['grams'] for item in items), 2),
        **nutrition.get('totals', {}),
    }
    return pd.DataFrame([*items, total_row])

print('Pipeline functions are ready.')


## Cell 6 — Open the app
The app uses a local URL, such as `http://127.0.0.1:7860`. If the browser does not open automatically, click the URL printed below.


In [ ]:
# === Cell 6: Gradio app ===

import json
import pandas as pd
import gradio as gr
from PIL import Image

if not (globals().get("CLIP_READY", False) and globals().get("LMSTUDIO_READY", False)):
    raise RuntimeError("Complete Cells 2 and 4 successfully before opening the app.")

print("Gradio version:", gr.__version__)

def run_pipeline(image, servings, calories_goal, dietary_prefs, grams_overrides_json):
    if image is None:
        raise gr.Error("Please upload an image before clicking Analyze & Create Recipe.")

    pil = image.convert("RGB")

    overrides = None
    if grams_overrides_json and grams_overrides_json.strip():
        try:
            overrides = json.loads(grams_overrides_json)
            if not isinstance(overrides, dict):
                raise ValueError("Overrides must be a JSON object.")
        except Exception as error:
            raise gr.Error(f"Invalid portion-overrides JSON: {error}")

    try:
        result = analyze_food_image(
            pil,
            user_prefs=dietary_prefs or "",
            servings=int(servings),
            calories_goal=int(calories_goal),
            grams_overrides=overrides,
        )
    except Exception as error:
        raise gr.Error(f"Food analysis failed: {error}") from error

    labels_json = json.dumps(result["detections"], indent=2, default=str)
    df = nutrition_df(result["nutrition"])
    recipe = result["recipe_text"]
    if result["nutrition"].get("skipped"):
        recipe += "\n\n**Nutrition lookup unavailable for:** " + ", ".join(result["nutrition"]["skipped"])
        recipe += ". These foods are excluded from the partial total."

    return labels_json, df, recipe


# Close an older copy if this cell is rerun.
try:
    gr.close_all()
except Exception:
    pass


with gr.Blocks(title="Simple Food Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## Simple Food Analyzer\n"
        "Upload a food photo, estimate nutrition, and generate a personalized recipe. CLIP suggests possible foods; check the labels yourself. Portions are assumed unless you enter gram amounts. Scores compare the candidate labels; they are not calibrated probabilities that an ingredient is present."
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_in = gr.Image(
                type="pil",
                label="Upload a food photo",
                height=320,
            )

            servings = gr.Slider(
                minimum=1,
                maximum=6,
                value=2,
                step=1,
                label="Servings",
            )

            calories_goal = gr.Slider(
                minimum=200,
                maximum=1200,
                value=600,
                step=50,
                label="Target Calories per Serving",
            )

            dietary_prefs = gr.Textbox(
                label="Dietary preferences or restrictions",
                placeholder="For example: high protein, no dairy",
            )

            grams_overrides = gr.Textbox(
                label="Optional portion overrides as JSON",
                placeholder='{"salmon": 180, "asparagus": 90}',
            )

            run_btn = gr.Button(
                "Analyze & Create Recipe",
                variant="primary",
            )

        with gr.Column(scale=1):
            labels_out = gr.Code(
                label="Top-3 food suggestions (relative scores)",
                language="json",
            )

            nutrition_out = gr.Dataframe(
                label="Estimated nutrition for the whole dish (assumed grams)",
                wrap=True,
                interactive=False,
            )

            recipe_out = gr.Markdown(
                label="Personalized Recipe"
            )

    run_btn.click(
        fn=run_pipeline,
        inputs=[
            image_in,
            servings,
            calories_goal,
            dietary_prefs,
            grams_overrides,
        ],
        outputs=[
            labels_out,
            nutrition_out,
            recipe_out,
        ],
        api_name=False,
    )


demo.launch(
    share=False,
    inline=False,
    inbrowser=True,
    prevent_thread_lock=True,
    show_error=True,
    show_api=False,
)


## Cell 7 — Optional local-photo test
This extra test is disabled during Run All. The app above is sufficient to use the lab.


In [ ]:
# === Cell 7: Optional test with a local photo ===
# Leave False during Run All. Change to True only when you want this extra run.
RUN_LOCAL_TEST = False

if RUN_LOCAL_TEST:
    from pathlib import Path
    from IPython.display import display

    sample_path = Path.home() / 'Desktop' / 'food05.jpg'
    if not sample_path.is_file():
        raise FileNotFoundError(f'Update sample_path to an existing photo: {sample_path}')
    with Image.open(sample_path) as opened:
        img = opened.convert('RGB')
    display(img)
    result = analyze_food_image(
        img, user_prefs='high protein, no dairy', servings=2, calories_goal=600,
    )
    print('Food suggestions:', result['detections'])
    display(nutrition_df(result['nutrition']))
    if result['nutrition'].get('skipped'):
        print('Excluded from nutrition totals:', result['nutrition']['skipped'])
    print(result['recipe_text'])
else:
    print('Optional local-photo test skipped. Use the Gradio page from Cell 6.')


## If a check stops

| Where it stops | What to do |
| --- | --- |
| Cell 1 prints Python 3.14 | Switch to **Python (Nutritionist 3.11)**. Repeating pip in the old kernel cannot fix its interpreter mismatch. |
| Cell 1 says a restart is needed | Restart the kernel, then Run All. A rerun without restarting does not clear loaded packages. |
| Cell 1 reports a pip failure | Read that installation error; confirm Python 3.11 and internet access. Setup never reports success after a failed pip command. |
| Cell 2 cannot connect | Start the LM Studio server, check port 1234 and authentication, then rerun Cell 2. |
| Cell 2 lists several models | Set `MODEL_ID` to the exact chat model ID printed by the server, then rerun Cell 2. |
| Cell 4 cannot load weights | Check Hugging Face access and free disk space, then rerun Cell 4. |
| Recipe generation fails after labels appear | Check LM Studio and the selected chat model. The CLIP dependency has already passed its checks. |

## Official references

- [Registering a separate conda/IPython kernel](https://ipython.readthedocs.io/en/stable/install/kernel_install.html)
- [PyTorch versioned installation instructions](https://pytorch.org/get-started/previous-versions/)
- [Transformers 4.57.1 package](https://pypi.org/project/transformers/4.57.1/)
- [Gradio 5.49.1 package](https://pypi.org/project/gradio/5.49.1/)
- [Official CLIP checkpoint](https://huggingface.co/openai/clip-vit-base-patch32)
- [LM Studio model-list endpoint](https://lmstudio.ai/docs/developer/openai-compat/models)
- [LM Studio chat-completion endpoint](https://lmstudio.ai/docs/developer/openai-compat/chat-completions)

The output cells are cleared so the old installation failures are not mistaken
for results from this revision.

**Validation:** This revision passed notebook-format checks, package imports,
real inference with the official CLIP weights, nutrition/portion checks, and
a live local Gradio launch and callback in an isolated Linux Python 3.11
environment using PyTorch 2.8.0 CPU. LM Studio model discovery and recipe
requests were tested against a simulated local HTTP server. Your Mac and
your actual LM Studio model have not been run remotely; Cells 1, 2, and 4
perform the relevant checks when you run this notebook there.
